MÔ PHỎNG VÀ TẠO DỮ LIỆU NHIỄU (ANOMALY SIMULATION)

Nhóm chủ động can thiệp vào các biến dữ liệu sạch trên RAM bằng cách tiêm (inject) các giá trị lỗi mang tính hệ thống nhằm thử thách quy trình xử lý dữ liệu (ETL):
*   **Giá trị khuyết thiếu (NaN):** Xuất hiện ngẫu nhiên từ $3\%$ đến $6\%$ trên các trường thông tin cốt lõi (Amount, Category, Credit Score,...).
*   **Sai lệch định dạng thời gian:** Giả lập lỗi đồng bộ với chuỗi lạ `"ERR_DATE_FORMAT_99"`.
*   **Dữ liệu trùng lặp (Duplicates):** Tạo $10\%$ bản ghi bị lặp hoàn toàn trong dữ liệu chi tiêu.
*   **Giá trị ngoại lai phi lý (Outliers):** Nhân số tiền lên $100$ lần cho $50$ giao dịch ngẫu nhiên, gán điểm tín dụng về mức cực đoan $-999.0$.
*   **Lỗi chính tả (Typos):** Làm bẩn nhãn danh mục (ví dụ: `Travel` thành `Travell_err`, `Grocery` thành `Grocerie$`).

In [ ]:
import pandas as pd
import numpy as np
import random
np.random.seed(42)
random.seed(42)
# NẠP DỮ LIỆU GỐC
try: 
    df_expenses_raw = pd.read_csv('Expenses_clean.csv')
    df_income_raw = pd.read_csv('Income_clean.csv')
    df_budget_raw = pd.read_csv('budget_data.csv')
    df_tracker_raw = pd.read_csv('personal_finance_tracker_dataset.csv')
    df_8000_raw = pd.read_csv('personal_finance_dataset_8000_extended.csv')
except FileNotFoundError as e:
    print(f" Thiếu file trên hệ thống. Vui lòng cập nhật file vào! {e}")
    raise
# GÂY NHIỄU CHO CÁC FILE DỮ LIỆU
#1. Gây nhiễu file Expenses - Income - Budget
# Tạo ô trống ngẫu nhiên ở cột số tiền - Amount và danh mục - Category
mask_exp_amt = np.random.rand(len(df_expenses_raw)) < 0.05
mask_exp_cat = np.random.rand(len(df_expenses_raw)) < 0.04
mask_inc_amt = np.random.rand(len(df_income_raw)) < 0.05
mask_inc_cat = np.random.rand(len(df_income_raw)) < 0.05
df_budget_raw.loc[np.random.rand(len(df_budget_raw)) < 0.03, 'amount'] = np.nan
df_expenses_raw.loc[mask_exp_amt, 'amount'] = np.nan
df_expenses_raw.loc[mask_exp_cat, 'category'] = np.nan
df_income_raw.loc[mask_inc_amt, 'amount'] = np.nan
df_income_raw.loc[mask_inc_cat, 'category'] = np.nan
# Gây lỗi định dạng ngày tháng
df_expenses_raw['date_time'] = (df_expenses_raw['date_time'].astype(str).apply(lambda x: "ERR_DATE_FORMAT_99" if random.random() < 0.03 else x)
)
df_income_raw['date_time'] = df_income_raw['date_time'].astype(str).apply(lambda x: "ERR_DATE_FORMAT_99" if random.random() < 0.04 else x
)
#Lỗi Trùng lặp
ty_le_trung_exp = 0.10  
so_dong_trung_exp = int(len(df_expenses_raw) * ty_le_trung_exp)
df_dup_exp = df_expenses_raw.sample(n = so_dong_trung_exp, random_state = 12)
df_exp_noisy = pd.concat([df_expenses_raw, df_dup_exp], ignore_index = True)
df_exp_noisy = df_exp_noisy.sample(frac = 1, random_state = 12).reset_index(drop = True)
#2. Gây nhiễu cho file 8000_extended
# Tạo giá trị trống khuyết thiếu ngẫu nhiên ở cột số tiền - Amount và danh mục - Category
mask_amt = np.random.rand(len(df_8000_raw)) < 0.05
mask_cat = np.random.rand(len(df_8000_raw)) < 0.04
df_8000_raw['Amount'] = df_8000_raw['Amount'].mask(mask_amt)
df_8000_raw['Category'] = df_8000_raw['Category'].mask(mask_cat)
# Tạo thêm giao dịch bất thường với dòng tiền lớn hơn thực tế
outlier_idx = np.random.choice(df_8000_raw.index, size = 50, replace = False)
df_8000_raw.loc[outlier_idx, 'Amount'] = df_8000_raw.loc[outlier_idx, 'Amount'] * 100
# Tạo lỗi kí tự và chính tả ở danh mục
df_8000_raw['Category'] = df_8000_raw['Category'].replace({'Travel': 'Travell_err', 'Grocery': 'Grocerie$', 'Bills': 'Biills'})
#3. Gây nhiễu cho file Trakcer_dataset
# Tạo giá trị trống hệ thống ở các đặc trưng cốt lõi
for col in ['monthly_income', 'credit_score', 'debt_to_income_ratio', 'financial_stress_level']:
    mask = np.random.rand(len(df_tracker_raw)) < 0.06
    df_tracker_raw[col] = df_tracker_raw[col].mask(mask)
# Giá trị vô lí cho điểm tín dụng
bad_idx = np.random.choice(df_tracker_raw.index, size = 15, replace = False)
df_tracker_raw.loc[bad_idx, 'credit_score'] = -999.0
# XUẤT CÁC FILE NHIỄU
df_income_raw.to_csv("Income_Noisy.csv", index = False)
df_8000_raw.to_csv("8000_Extended_Noisy.csv", index = False)
df_budget_raw.to_csv("Budget_Noisy.csv", index = False)
df_tracker_raw.to_csv("Tracker_Noisy.csv", index = False)
df_expenses_raw.to_csv("Expenses_Noisy.csv", index = False)  


KHÁM PHÁ DỮ LIỆU THÔ VÀ DÒ TÌM NHIỄU (DATA EXPLORATION & ANOMALY DETECTION)

Sau khi tạo dữ liệu nhiễu, hệ thống tiến hành nạp lại các tệp dữ liệu đã bị làm bẩn để thực hiện bước "khám phá chẩn đoán". Mục tiêu của bước này bao gồm:
1.  **Đánh giá tổng quan cấu trúc:** Kiểm tra kích thước (shape) và kiểu dữ liệu (info) của từng bảng.
2.  **Xác định tỷ lệ khuyết thiếu (NaN):** Đo lường mức độ mất mát thông tin trên các trường dữ liệu quan trọng.
3.  **Phát hiện trùng lặp (Duplicates):** Xác định các bản ghi bị nhân bản gây nhiễu số liệu.
4.  **Dò tìm các điểm bất thường (Anomalies):** 
    *   Phát hiện các lỗi gõ sai chính tả và ký tự lạ trong cột phân loại (`Category`).
    *   Phát hiện các giá trị âm vô lý ở chỉ số điểm tín dụng (`credit_score`).
    *   Phát hiện lỗi định dạng cấu trúc ngày tháng (`date_time`).

In [ ]:
df_8000 = pd.read_csv("8000_Extended_Noisy.csv")
df_tracker = pd.read_csv("Tracker_Noisy.csv")
df_budget = pd.read_csv("Budget_Noisy.csv")
df_expenses = pd.read_csv("Expenses_Noisy.csv")
df_income = pd.read_csv("Income_Noisy.csv")
# Xem kích thước dữ liệu
print("\n KÍCH THƯỚC DỮ LIỆU")
print("8000:", df_8000.shape)
print("Tracker:", df_tracker.shape)
print("Budget:", df_budget.shape)
print("Expenses:", df_expenses.shape)
print("Income:", df_income.shape)
print("-" * 70)
# Xem thông tin dữ liệu
print("\n THÔNG TIN DỮ LIỆU")
df_8000.info()
df_tracker.info()
df_budget.info()
df_expenses.info()
df_income.info()
print("-" * 70)
# Xem thống kê mô tả dữ liệu
print("\n THỐNG KÊ MÔ TẢ")
display(df_8000.describe().round(2)) #Thống kê mô tả cột số tiền
display(df_tracker.describe().round(2)) #Thống kê mô tả các chỉ số tài chính
print("-" * 70)
# Kiểm tra giá trị khuyết trống thiếu
print("\n GIÁ TRỊ KHUYẾT TRỐNG THIẾU")
print(f"File 8000: \n{df_8000.isnull().sum()}\n")
print(f"File Tracker: \n{df_tracker.isnull().sum()}\n")
print(f"File Budget: \n{df_budget.isnull().sum()}\n")
print(f"File Exp: \n{df_expenses.isnull().sum()}\n")
print(f"File Inc: \n{df_income.isnull().sum()}")
print("-" * 70)
# Kiểm tra trùng lặp
print("\n DỮ LIỆU TRÙNG LẶP")
print(f"Trùng lặp file 8000: {df_8000.duplicated().sum()}")
print(f"Trùng lặp file Tracker: {df_tracker.duplicated().sum()}")
print(f"Trùng lặp file Budget: {df_budget.duplicated().sum()}")
print(f"Trùng lặp file Exp: {df_expenses.duplicated().sum()}")
print(f"Trùng lặp file Inc: {df_income.duplicated().sum()}") 
print("-" * 70)
# Kiểm tra các danh mục chi tiêu
print("\n CATEGORY BẤT THƯỜNG")
print(df_8000["Category"].value_counts())
print("-" * 70)
# Kiểm tra Credit Score bất thường
print("\n CREDIT SCORE BẤT THƯỜNG")
print("Số lượng:", (df_tracker["credit_score"] < 0).sum())
print("-" * 70)
# Khám phá dữ liệu ngày tháng
print("\n DỮ LIỆU NGÀY THÁNG")
print(df_expenses["date_time"].head())
print(df_income["date_time"].head())

LÀM SẠCH DỮ LIỆU THU - CHI VÀ NGÂN SÁCH (DATA CLEANING & ETL - PART 1)

Sau khi hoàn tất giai đoạn chẩn đoán lỗi, hệ thống tiến hành xây dựng quy trình ETL (Extract - Transform - Load) giai đoạn 1 để làm sạch các tệp dữ liệu cốt lõi: `Expenses`, `Income` và `Budget`. Các kỹ thuật làm sạch được áp dụng bao gồm:
1.  **Loại bỏ dữ liệu trùng lặp (Deduplication):** Sử dụng phương thức `.drop_duplicates()` để giải quyết triệt để các bản ghi bị lặp 10% đã tạo.
2.  **Chuẩn hóa dữ liệu chuỗi thời gian:** Chuyển đổi cột `date_time` về đúng định dạng chuẩn `datetime64`. Với các dòng bị lỗi cấu trúc chủ động (`"ERR_DATE_FORMAT_99"`), hệ thống ép kiểu lỗi về dạng khuyết thiếu `NaT` (thông qua tham số `errors='coerce'`) và xử lý bằng kỹ thuật điền tiến/lùi liền kề (`ffill().bfill()`) nhằm duy trì tính liên tục của trục thời gian.
3.  **Xử lý giá trị khuyết thiếu (Imputation):**
    *   Đối với các thuộc tính định lượng số tiền (`amount`), áp dụng phương pháp điền khuyết bằng **Trung vị (Median)** để hạn chế tối đa ảnh hưởng từ các giá trị cực trị lệch.
    *   Đối với các thuộc tính định danh phân loại (`category`), áp dụng phương pháp điền khuyết bằng **Yếu vị (Mode)** - giá trị xuất hiện nhiều nhất của cột đó.

In [ ]:
import pandas as pd
import numpy as np  
df_budget = pd.read_csv("Budget_Noisy.csv")
df_expenses = pd.read_csv("Expenses_Noisy.csv")
df_income = pd.read_csv("Income_Noisy.csv")
# Xóa dòng trùng lặp 
df_expenses.drop_duplicates(inplace = True)
# Kiểm tra trùng lặp
print(f"Trùng lặp file Exp: {df_expenses.duplicated().sum()}")
# Định dạng ngày tháng
df_expenses['date_time'] = pd.to_datetime(df_expenses['date_time'], errors='coerce').ffill().bfill()
df_income['date_time'] = pd.to_datetime(df_income['date_time'], errors='coerce').ffill().bfill()
# Kiểm tra ngày tháng
print("\n DỮ LIỆU NGÀY THÁNG")
print(df_expenses["date_time"].head())
print(df_income["date_time"].head())
# Xử lí cac giá trị bị khuyết trống thiếu ở cột Amount với giá trị ở giữa (Median) - ở cột Category với chữ xuất hiện nhiều nhất (Mode) 
df_expenses['amount'] = df_expenses['amount'].fillna(df_expenses['amount'].median())
df_income['amount'] = df_income['amount'].fillna(df_income['amount'].median())
df_budget['amount'] = df_budget['amount'].fillna(df_budget['amount'].median())
df_expenses['category'] = df_expenses['category'].fillna(df_expenses['category'].mode()[0])
df_income['category'] = df_income['category'].fillna(df_income['category'].mode()[0])
# Kiểm tra các giá trị khuyết thiếu
print("\n GIÁ TRỊ KHUYẾT TRỐNG THIẾU")
print(f"File Budget: \n{df_budget.isnull().sum()}\n")
print(f"File Exp: \n{df_expenses.isnull().sum()}\n")
print(f"File Inc: \n{df_income.isnull().sum()}")
# XUẤT FILE CLEANED
df_budget.to_csv("Budget_CLEANED.csv", index = False)
df_income.to_csv("Income_CLEANED.csv", index = False)
df_expenses.to_csv("Expenses_CLEANED.csv", index = False )

LÀM SẠCH DỮ LIỆU CHỈ SỐ TÀI CHÍNH VÀ THEO DÕI CÁ NHÂN (DATA CLEANING & ETL - PART 2)
Trong giai đoạn này, hệ thống tập trung xử lý tệp dữ liệu theo dõi tài chính cá nhân (`Tracker_Noisy.csv`) để chuẩn bị làm đầu vào cho các thuật toán phân tích và học máy. Quy trình xử lý bao gồm các kỹ thuật nâng cao sau:
1.  **Xử lý lỗi logic hệ thống:** Chuyển đổi các giá trị điểm tín dụng âm phi lý (`-999.0`) về dạng khuyết thiếu `NaN` để tránh làm lệch phân phối dữ liệu.
2.  **Khôi phục giá trị khuyết thiếu (Imputation):**
    *   Với các biến định lượng liên tục (`monthly_income`, `credit_score`, `debt_to_income_ratio`), áp dụng phương pháp điền khuyết bằng **Trung vị (Median)**.
    *   Với biến phân loại cấp bậc mức độ căng thẳng tài chính (`financial_stress_level`), áp dụng phương pháp điền khuyết bằng **Yếu vị (Mode)**.
3.  **Lọc nhiễu và xử lý giá trị ngoại lai (Outlier Treatment using IQR):** 
    *   Sử dụng phương pháp **Hàng rào Tukey (Interquartile Range - IQR)** với ngưỡng $[Q_1 - 1.5 \times IQR, Q_3 + 1.5 \times IQR]$ để xác định các giá trị thu nhập (`monthly_income`) bất thường.
    *   Để bảo toàn kích thước mẫu dữ liệu, các điểm ngoại lai này sẽ được "làm mịn" (smooth) bằng cách thay thế trực tiếp bằng giá trị **Trung vị (Median)** thay vì xóa bỏ.

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Đọc dữ liệu

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
csv_path = os.path.join(BASE_DIR, "Tracker_Noisy.csv")

df_track = pd.read_csv(csv_path)

print("========== THÔNG TIN BAN ĐẦU ==========")
print(df_track.info())
print("\nKích thước dữ liệu:", df_track.shape)

# 2. Thay giá trị vô lý
# credit_score = -999 -> NaN

df_track["credit_score"] = df_track["credit_score"].replace(-999.0, np.nan)

# 3. Điền giá trị thiếu

numeric_cols = [
    "monthly_income",
    "credit_score",
    "debt_to_income_ratio"
]

for col in numeric_cols:
    df_track[col] = df_track[col].fillna(df_track[col].median())

df_track["financial_stress_level"] = (
    df_track["financial_stress_level"]
    .fillna(df_track["financial_stress_level"].mode()[0])
)

# 4. Xử lý Outlier bằng IQR
# (Thay bằng Median)

Q1 = df_track["monthly_income"].quantile(0.25)
Q3 = df_track["monthly_income"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

median_income = df_track["monthly_income"].median()

outlier_count = (
    (df_track["monthly_income"] < lower) |
    (df_track["monthly_income"] > upper)
).sum()

print("\nSố Outlier của monthly_income:", outlier_count)

df_track.loc[
    (df_track["monthly_income"] < lower) |
    (df_track["monthly_income"] > upper),
    "monthly_income"
] = median_income

# 5. Kiểm tra sau khi clean

print("\n========== GIÁ TRỊ THIẾU ==========")
print(df_track.isnull().sum())

print("\n========== THỐNG KÊ ==========")
print(df_track.describe(include="all"))

print("\n========== THÔNG TIN ==========")
print(df_track.info())

# 6. Lưu dữ liệu sạch

output_path = os.path.join(BASE_DIR, "Tracker_Clean.csv")

df_track.to_csv(output_path, index=False)

print("\n===================================")
print("Đã làm sạch dữ liệu thành công!")
print("File được lưu tại:")
print(output_path)
print("===================================")

LÀM SẠCH TẬP DỮ LIỆU GIAO DỊCH MỞ RỘNG (DATA CLEANING & ETL - PART 3)

Trong phần này, hệ thống sẽ thực hiện làm sạch tệp dữ liệu giao dịch mở rộng (`8000_Extended_Noisy.csv`). Đây là một tập dữ liệu lớn chứa các lỗi hỗn hợp bao gồm cả lỗi nhập liệu văn bản (chính tả) lẫn lỗi hệ thống (ngoại lai số tiền lớn). 

Quy trình xử lý chuẩn hóa bao gồm các kỹ thuật:
1.  **Chuẩn hóa dữ liệu định danh (Text Categorical Cleaning):** Sử dụng phương thức `.replace()` để ánh xạ ngược các nhãn danh mục bị viết sai chính tả hoặc dính ký tự đặc biệt (`Travell_err`, `Grocerie$`, `Biills`) về các nhãn chuẩn (`Travel`, `Grocery`, `Bills`).
2.  **Khôi phục giá trị khuyết thiếu (Imputation):**
    *   Điền khuyết cho cột danh mục (`Category`) bằng **Yếu vị (Mode)** - giá trị xuất hiện phổ biến nhất.
    *   Điền khuyết cho cột số tiền giao dịch (`Amount`) bằng **Trung vị (Median)** để tránh sai số do phân phối lệch.
3.  **Xử lý và làm mịn giá trị ngoại lai (Outlier Smoothing using IQR):**
    *   Sử dụng phương pháp **Hàng rào Tukey (Interquartile Range - IQR)** để cô lập 50 giao dịch bất thường bị nhân khống số tiền lên gấp 100 lần ở Bước 2.
    *   Gán các giá trị ngoại lai vượt ngưỡng $[Q_1 - 1.5 \times IQR, Q_3 + 1.5 \times IQR]$ về mức giá trị **Trung vị (Median)** của toàn bộ tập dữ liệu nhằm đảm bảo tính ổn định cho các bước phân tích tương quan phía sau.

In [ ]:
import pandas as pd
import numpy as np
import os

# Đọc file CSV nằm cùng thư mục với file .py
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
csv_path = os.path.join(BASE_DIR, "8000_Extended_Noisy.csv")

df_8000 = pd.read_csv(csv_path)

# 1. Sửa lỗi chính tả Category

df_8000["Category"] = df_8000["Category"].replace({
    "Travell_err": "Travel",
    "Grocerie$": "Grocery",
    "Biills": "Bills"
})

# 2. Xử lý giá trị thiếu

# Category -> thay bằng giá trị xuất hiện nhiều nhất
df_8000["Category"] = df_8000["Category"].fillna(
    df_8000["Category"].mode()[0]
)

# Amount -> thay bằng trung vị
df_8000["Amount"] = df_8000["Amount"].fillna(
    df_8000["Amount"].median()
)

# 3. Xử lý Outlier Amount

Q1 = df_8000["Amount"].quantile(0.25)
Q3 = df_8000["Amount"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

median_amount = df_8000["Amount"].median()

df_8000.loc[
    (df_8000["Amount"] < lower_bound) |
    (df_8000["Amount"] > upper_bound),
    "Amount"
] = median_amount

# 4. Lưu file sạch

output_path = os.path.join(BASE_DIR, "8000_Extended_Clean.csv")
df_8000.to_csv(output_path, index=False)

print("Đã làm sạch dữ liệu!")
print("File lưu tại:", output_path)

TỰ ĐỘNG HÓA THIẾT KẾ VÀ KHỞI TẠO CƠ SỞ DỮ LIỆU MYSQL (DATABASE LOADING & INTEGRATION)

Sau khi hoàn thành chuỗi làm sạch dữ liệu của toàn bộ 5 tệp nguồn, hệ thống tiến hành bước chuyển đổi và nạp dữ liệu (Load - giai đoạn cuối của quy trình ETL). Thay vì lưu trữ tệp tin tĩnh đơn thuần, nhóm nghiên cứu tích hợp hệ quản trị cơ sở dữ liệu quan hệ **MySQL** để quản lý dữ liệu tập trung.

Quy trình tự động hóa được xây dựng thông qua một Class trong Python với các ưu điểm:
1.  **Lập trình hướng đối tượng (OOP):** Đóng gói cấu hình kết nối, tăng tính tái sử dụng và độ bảo mật của mã nguồn.
2.  **Kết hợp tối ưu công nghệ:** Sử dụng `mysql-connector` để thiết lập kênh giao tiếp, đồng thời dùng `SQLAlchemy Engine` nhằm tối ưu hóa tiến trình nạp (Bulk Insert) hàng ngàn bản ghi một cách nhanh chóng.
3.  **Hàng rào kiểm soát chất lượng cuối cùng (Quality Gate):** Trước khi nạp vào hệ thống Database, mã nguồn thực hiện quét loại bỏ các dòng trống hoàn toàn (`.dropna(how='all')`) và lọc trùng lặp bổ sung (`.drop_duplicates()`) để đảm bảo tính toàn vẹn của cơ sở dữ liệu.

In [ ]:
import pandas as pd
import mysql.connector
from mysql.connector import Error
from sqlalchemy import create_engine

class Connection_DB:
    def __init__(self):
        # Cấu hình tài khoản theo mẫu
        self.db_config = {
            'host': 'localhost',
            'user': 'root',
            'password': '1234',  
            'database': 'finance_db'
        }
    def connect_db(self):
        connection = None
        try:
            connection = mysql.connector.connect(**self.db_config)
            if connection.is_connected():
                print(" Kết nối MySQL thành công!")
                datasets = {
                    "Income_CLEANED": "Income_CLEANED.csv",
                    "Expenses_CLEANED": "Expenses_CLEANED.csv",
                    "Budget_CLEANED": "Budget_CLEANED.csv",
                    "Tracker_CLEANED": "Tracker_CLEANED.csv",
                    "8000_Extended_Clean": "8000_Extended_Clean.csv"
                }
                engine = create_engine(f"mysql+mysqlconnector://{self.db_config['user']}:{self.db_config['password']}@{self.db_config['host']}/{self.db_config['database']}")
                # Quét trùng và nạp bảng dữ liệu nhanh
                for table_name, file_path in datasets.items():
                    df = pd.read_csv(file_path).dropna(how='all').drop_duplicates()
                    df.to_sql(name=table_name, con=engine, if_exists='replace', index=False)
        except Error as e:
            print(f" Lỗi: {e}")
        finally:
            if connection and connection.is_connected():
                connection.close()
                print(" Đã ngắt kết nối an toàn.")
if __name__ == "__main__":
    db = Connection_DB()
    db.connect_db()

TRỰC QUAN HÓA VÀ PHÂN TÍCH CHỈ SỐ TÀI CHÍNH CHUYÊN SÂU (ADVANCED VISUALIZATION & INSIGHTS)

Giai đoạn này hiện thực hóa toàn bộ các nỗ lực làm sạch dữ liệu trước đó bằng cách chuyển hóa dữ liệu thô thành **Trí tuệ kinh doanh (Business Intelligence)**. Nhóm nghiên cứu thiết lập một hệ thống phân tích trực quan hóa đa chiều gồm 7 biểu đồ chuyên sâu và xuất tự động các tệp báo cáo thống kê:

1.  **Cán cân thanh khoản tiền mặt ngắn hạn (Column Chart):** Đánh giá sức khỏe dòng tiền thông qua mối tương quan giữa Thu nhập - Chi tiêu - Dòng tiền ròng cùng Chỉ số tiêu dùng (Consumption Ratio).
2.  **Biên độ lệch ròng giữa Thực chi và Ngân sách (Diverging Bar Chart):** Nhận diện trực quan các danh mục kiểm soát tốt (dưới ngân sách - màu xanh) và các điểm nóng tài chính (vượt ngân sách - màu đỏ).
3.  **Danh mục lũy kế tiêu tốn tiền mặt dài hạn (Horizontal Bar Chart):** Áp dụng kỹ thuật chuyển màu Gradient để làm nổi bật Top 10 danh mục "ngốn tiền" nhiều nhất trong dài hạn cùng tỷ trọng tương đối.
4.  **Biên độ chi tiêu theo áp lực tâm lý và cấu trúc nợ (Boxplot song hành):** Khảo sát hành vi chi tiêu linh hoạt (Discretionary Spending) của các nhóm đối tượng dựa trên ma trận rủi ro: Mức độ Stress (Low/Medium/High) kết hợp Hàng rào cảnh báo nợ an toàn $DTI = 35\%$.
5.  **Ma trận tương quan tuyến tính đa biến (Heatmap):** Tính toán hệ số tương quan Pearson giữa tất cả các cặp biến số tài chính để tìm ra các quy luật chuyển dịch ngầm.
6.  **Mật độ phân phối điểm tín nhiệm (KDE Plot):** Dựng đường mật độ xác suất liên tục của điểm tín dụng toàn hệ thống, cô lập phân vùng điểm "dưới chuẩn" ($Credit < 600$) để cảnh báo tỷ lệ rủi ro tín dụng nợ xấu.
7.  **Tương quan thu nhập với bẫy lạm phát lối sống (Scatter Plot):** Kiểm chứng giả thuyết "Thu nhập tăng thì chi tiêu tăng" thông qua mô hình hồi quy tuyến tính thực nghiệm và phân tách nhóm đạt/không đạt mục tiêu tiết kiệm.

In [ ]:
# ============================================================
# BÀI PHÂN TÍCH & TRỰC QUAN HÓA DỮ LIỆU TÀI CHÍNH
# File này đọc dữ liệu đã làm sạch, xử lý dữ liệu,
# tạo 7 biểu đồ, xuất 7 bảng CSV và ghi file tóm tắt kết quả.
# Các comment bên dưới giải thích rõ từng nhóm lệnh chính.
# ============================================================

# Import pandas để đọc file CSV, xử lý bảng dữ liệu, groupby, merge và xuất CSV.
import pandas as pd
# Import numpy để xử lý số học, điều kiện np.where, giá trị NaN và vô cực.
import numpy as np
# Import matplotlib.pyplot để vẽ và lưu các biểu đồ.
import matplotlib.pyplot as plt
# Import Path để quản lý đường dẫn thư mục/file rõ ràng hơn.
from pathlib import Path
# Import Patch để tự tạo chú thích màu trong biểu đồ.
from matplotlib.patches import Patch
# Khai báo thư mục đầu ra để lưu toàn bộ biểu đồ, bảng CSV và file tóm tắt.
out = Path("Financial_Visualization_Youngbillydasixx")
# Tạo thư mục đầu ra nếu chưa có; nếu đã có thì không báo lỗi.
out.mkdir(exist_ok=True)

# Đọc file dữ liệu thu nhập đã làm sạch.
income = pd.read_csv("Income_CLEANED.csv")
# Đọc file dữ liệu chi tiêu đã làm sạch.
expenses = pd.read_csv("Expenses_CLEANED.csv")
# Đọc file dữ liệu ngân sách đã làm sạch.
budget = pd.read_csv("Budget_CLEANED.csv")
# Đọc file theo dõi tài chính, gồm các chỉ số như thu nhập tháng, chi tiêu, stress, DTI, credit score.
tracker = pd.read_csv("Tracker_CLEANED.csv")
# Đọc file dữ liệu mở rộng 8000 dòng để phân tích chi tiêu dài hạn.
extended = pd.read_csv("8000_Extended_Clean.csv")

# Thiết lập font chữ chung cho biểu đồ để hỗ trợ hiển thị tiếng Việt.
plt.rcParams["font.family"] = "DejaVu Sans"
# Thiết lập cỡ chữ tiêu đề biểu đồ.
plt.rcParams["axes.titlesize"] = 14
# Thiết lập cỡ chữ nhãn trục X và Y.
plt.rcParams["axes.labelsize"] = 11

# Định nghĩa hàm lưu biểu đồ để dùng lại nhiều lần cho 7 biểu đồ.
def save_chart(file_name):
    plt.tight_layout()
    plt.savefig(out / file_name, dpi=300, bbox_inches="tight")
    plt.close()

# Định nghĩa hàm chuẩn hóa tên danh mục chi tiêu/ngân sách trước khi so sánh.
def normalize_category(value):
    return str(value).strip().lower().replace("restuarant", "restaurant").replace("coffe", "cafe")

# Danh sách các cột cần chuyển sang kiểu số để tính toán và vẽ biểu đồ.
numeric_columns = [
    "monthly_income", "monthly_expense_total", "savings_rate", "budget_goal",
    "credit_score", "debt_to_income_ratio", "loan_payment", "investment_amount",
    "emergency_fund", "transaction_count", "discretionary_spending",
    "essential_spending", "rent_or_mortgage", "actual_savings", "savings_goal_met",
]
# Duyệt qua từng cột trong danh sách cần chuyển đổi kiểu dữ liệu.
for col in numeric_columns:
    if col in tracker.columns:
        tracker[col] = pd.to_numeric(tracker[col], errors="coerce")

# ============================================================
# 1. CÁN CÂN THANH KHOẢN TIỀN MẶT NGẮN HẠN
# Mục tiêu: tính tổng thu nhập, tổng chi tiêu, dòng tiền ròng và tỷ lệ tiêu dùng.
# ============================================================
# 1. CÁN CÂN THANH KHOẢN TIỀN MẶT NGẮN HẠN
# Tính tổng thu nhập bằng cách cộng toàn bộ cột amount trong bảng income.
total_income = income["amount"].sum()
# Tính tổng chi tiêu bằng cách cộng toàn bộ cột amount trong bảng expenses.
total_expenses = expenses["amount"].sum()
# Tính dòng tiền ròng = tổng thu nhập - tổng chi tiêu.
net_cash_flow = total_income - total_expenses
# Tính tỷ lệ tiêu dùng, cho biết chi tiêu chiếm bao nhiêu phần trăm thu nhập.
consumption_ratio = total_expenses / total_income * 100

# Tạo bảng kết quả tóm tắt để lưu các chỉ số chính ra CSV.
pd.DataFrame({
    "metric": ["Tổng thu nhập", "Tổng chi tiêu", "Dòng tiền ròng", "Tỷ lệ tiêu dùng so với thu nhập (%)"],
    "value": [total_income, total_expenses, net_cash_flow, consumption_ratio]
}).to_csv(out / "01_can_can_thanh_khoan.csv", index=False, encoding="utf-8-sig")

# Tạo khung biểu đồ với kích thước 9 x 5.5 inch.
plt.figure(figsize=(9, 5.5))
# Khai báo nhãn cho 3 cột trên biểu đồ.
labels = ["Tổng thu nhập", "Tổng chi tiêu", "Dòng tiền ròng"]
# Khai báo giá trị tương ứng với từng cột.
values = [total_income, total_expenses, net_cash_flow]
# Gán màu: xanh lá cho thu nhập, đỏ cho chi tiêu, xanh dương cho dòng tiền ròng.
colors = ["#3ad13a", "#e73737", "#3283be"]
# Vẽ biểu đồ cột từ labels và values.
bars = plt.bar(labels, values, color=colors)
# Đặt tiêu đề biểu đồ mục 1 và in đậm.
plt.title(" CÁN CÂN THANH KHOẢN TIỀN MẶT NGẮN HẠN", fontweight="bold")
# Đặt tên trục Y là Số tiền.
plt.ylabel("Số tiền")
# Duyệt qua từng cột để ghi giá trị lên đầu cột.
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:,.0f}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.text(0.5, max(values) * 0.82, f"Tỷ lệ tiêu dùng: {consumption_ratio:.2f}%",
         ha="center", bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))
plt.legend(handles=[
    Patch(color="#3ad13a", label=" Thu nhập"),
    Patch(color="#e73737", label=" Chi tiêu"),
    Patch(color="#3283be", label=" Dòng tiền ròng"),
], loc="upper right")
save_chart("01_column_chart_can_can_thanh_khoan.png")

# ============================================================
# 2. BIÊN ĐỘ LỆCH RÒNG GIỮA THỰC CHI VỚI HẠN MỨC NGÂN SÁCH
# Mục tiêu: so sánh ngân sách đặt ra và chi tiêu thực tế theo từng danh mục.
# ============================================================
# 2. BIÊN ĐỘ LỆCH RÒNG GIỮA THỰC CHI VỚI HẠN MỨC NGÂN SÁCH
# Chuẩn hóa tên danh mục trong bảng chi tiêu để gộp dữ liệu chính xác hơn.
expenses["category_norm"] = expenses["category"].map(normalize_category)
# Chuẩn hóa tên danh mục trong bảng ngân sách.
budget["category_norm"] = budget["category"].map(normalize_category)
# Tính tổng chi tiêu thực tế theo từng danh mục.
actual_spending = expenses.groupby("category_norm", as_index=False)["amount"].sum().rename(columns={"amount": "actual_spending"})
# Tính tổng ngân sách đặt ra theo từng danh mục.
budget_amount = budget.groupby("category_norm", as_index=False)["amount"].sum().rename(columns={"amount": "budget_amount"})
# Gộp ngân sách và thực chi theo danh mục; thay giá trị thiếu bằng 0.
budget_compare = budget_amount.merge(actual_spending, on="category_norm", how="outer").fillna(0)
# Tính độ lệch ngân sách = ngân sách - thực chi.
budget_compare["variance"] = budget_compare["budget_amount"] - budget_compare["actual_spending"]
# Gắn nhãn vượt ngân sách nếu variance âm, chi dưới ngân sách nếu variance dương.
budget_compare["status"] = np.where(budget_compare["variance"] < 0, "Vượt ngân sách", "Chi dưới ngân sách")
# Sắp xếp theo độ lệch để biểu đồ diverging bar dễ đọc.
budget_compare = budget_compare.sort_values("variance")
budget_compare.to_csv(out / "02_do_lech_ngan_sach.csv", index=False, encoding="utf-8-sig")

plt.figure(figsize=(10, max(7, len(budget_compare) * 0.33)))
plt.barh(budget_compare["category_norm"], budget_compare["variance"],
         color=np.where(budget_compare["variance"] >= 0, "#3ad13a", "#e73737"))
plt.axvline(0, color="black", linewidth=1)
plt.title(" BIÊN ĐỘ LỆCH RÒNG GIỮA THỰC CHI VỚI HẠN MỨC NGÂN SÁCH", fontweight="bold")
plt.xlabel("Ngân sách - Thực chi")
plt.ylabel("Danh mục")
plt.legend(handles=[
    Patch(color="#3ad13a", label=" Chi dưới ngân sách"),
    Patch(color="#e73737", label=" Vượt ngân sách"),
], loc="lower right")
save_chart("02_diverging_bar_chart_ngan_sach.png")

# ============================================================
# 3. DANH MỤC LŨY KẾ TIÊU TỐN TIỀN MẶT DÀI HẠN
# Mục tiêu: tìm các danh mục chi tiêu chiếm tỷ trọng lớn nhất trong dài hạn.
# ============================================================
# 3. DANH MỤC LŨY KẾ TIÊU TỐN TIỀN MẶT DÀI HẠN
# Tạo bản sao dữ liệu extended để xử lý riêng mà không làm thay đổi dữ liệu gốc.
extended_expense = extended.copy()
# Kiểm tra dữ liệu có cột loại giao dịch hay không.
if "TransactionType" in extended_expense.columns:
    temp = extended_expense[
        extended_expense["TransactionType"].astype(str).str.lower().str.contains("expense|debit|spent|payment", na=False)
    ]
    if not temp.empty:
        extended_expense = temp

# Tính tổng số tiền theo từng danh mục và sắp xếp giảm dần.
long_term = extended_expense.groupby("Category", as_index=False)["Amount"].sum().sort_values("Amount", ascending=False)
# Tính tỷ trọng phần trăm của từng danh mục trên tổng chi tiêu.
long_term["share_percent"] = long_term["Amount"] / long_term["Amount"].sum() * 100
long_term.to_csv(out / "03_tich_luy_dai_han_theo_danh_muc.csv", index=False, encoding="utf-8-sig")

# Lấy 10 danh mục có tổng chi tiêu cao nhất để vẽ biểu đồ.
top_long_term = long_term.head(10)
gradient_colors = [plt.cm.Blues(0.9 - i * (0.45 / max(len(top_long_term) - 1, 1))) for i in range(len(top_long_term))]
plt.figure(figsize=(10, 6))
bars = plt.barh(top_long_term["Category"], top_long_term["Amount"], color=gradient_colors)
plt.gca().invert_yaxis()
plt.title(" DANH MỤC LŨY KẾ TIÊU TỐN TIỀN MẶT DÀI HẠN", fontweight="bold")
plt.xlabel("Tổng chi tiêu tích lũy")
plt.ylabel("Danh mục")
for bar, value, percent in zip(bars, top_long_term["Amount"], top_long_term["share_percent"]):
    plt.text(value, bar.get_y() + bar.get_height() / 2, f" {value / 1e6:.2f} ({percent:.1f}%)",
             va="center", fontsize=9)
save_chart("03_bar_chart_tieu_dung_tich_luy.png")

# ============================================================
# 4. BIÊN ĐỘ CHI TIÊU THEO BIẾN ĐỊNH TÍNH ÁP LỰC VÀ NỢ
# Mục tiêu: so sánh chi tiêu linh hoạt theo mức stress và cấu trúc nợ.
# ============================================================
# 4. BIÊN ĐỘ CHI TIÊU THEO BIẾN ĐỊNH TÍNH ÁP LỰC VÀ NỢ
# Lấy 3 cột cần thiết cho biểu đồ boxplot: stress, tỷ lệ nợ/thu nhập và chi tiêu linh hoạt.
df4 = tracker[["financial_stress_level", "debt_to_income_ratio", "discretionary_spending"]].copy()
# Loại bỏ giá trị vô cực và giá trị thiếu để boxplot không bị lỗi.
df4 = df4.replace([np.inf, -np.inf], np.nan).dropna()
# Chuẩn hóa tên mức độ stress về dạng Low, Medium, High.
df4["financial_stress_level"] = df4["financial_stress_level"].astype(str).str.strip().str.capitalize()

# Thiết lập thứ tự mức stress theo mẫu báo cáo.
stress_order = ["Low", "High", "Medium"]
df4 = df4[df4["financial_stress_level"].isin(stress_order)].copy()
# Đặt ngưỡng DTI là 35% để phân loại nợ an toàn/nguy hiểm.
debt_threshold = 0.35
# Phân loại cấu trúc nợ: DTI > 35% là nguy hiểm, ngược lại là an toàn.
df4["debt_structure"] = np.where(df4["debt_to_income_ratio"] > debt_threshold, "Nguy hiểm (>35%)", "An toàn (<=35%)")

# Tạo bảng thống kê theo từng cặp mức stress và cấu trúc nợ.
summary_4 = df4.groupby(["financial_stress_level", "debt_structure"]).agg(
    so_luong=("discretionary_spending", "count"),
    low_q1=("discretionary_spending", lambda x: x.quantile(0.25)),
    trung_binh=("discretionary_spending", "mean"),
    trung_vi=("discretionary_spending", "median"),
    high_q3=("discretionary_spending", lambda x: x.quantile(0.75)),
    min_value=("discretionary_spending", "min"),
    max_value=("discretionary_spending", "max"),
).reset_index()
summary_4.to_csv(out / "04_so_sanh_nhom_rui_ro.csv", index=False, encoding="utf-8-sig")

red = "#d62728"
blue = "#1f77b4"
# Tạo các danh sách để lưu vị trí, dữ liệu và màu sắc cho boxplot.
positions, box_data, box_colors = [], [], []
# offset giúp hai boxplot trong cùng một mức stress nằm cạnh nhau.
offset = 0.18
# Duyệt từng mức stress để lấy dữ liệu nhóm nguy hiểm và nhóm an toàn.
for i, stress in enumerate(stress_order, start=1):
    dangerous_data = df4[(df4["financial_stress_level"] == stress) & (df4["debt_structure"] == "Nguy hiểm (>35%)")]["discretionary_spending"].dropna()
    safe_data = df4[(df4["financial_stress_level"] == stress) & (df4["debt_structure"] == "An toàn (<=35%)")]["discretionary_spending"].dropna()
    box_data.extend([dangerous_data, safe_data])
    positions.extend([i - offset, i + offset])
    box_colors.extend([red, blue])

# Tạo khung biểu đồ lớn hơn để hiển thị 6 boxplot rõ ràng.
plt.figure(figsize=(12, 7))
# Vẽ boxplot cho các nhóm chi tiêu linh hoạt.
box = plt.boxplot(
    box_data, positions=positions, widths=0.32, patch_artist=True, showfliers=True,
    medianprops=dict(color="#4d4d4d", linewidth=1.7),
    whiskerprops=dict(color="#4d4d4d", linewidth=1.2),
    capprops=dict(color="#4d4d4d", linewidth=1.2),
    flierprops=dict(marker="o", markerfacecolor="white", markeredgecolor="#4d4d4d", markersize=6)
)
# Gán màu đỏ/xanh cho từng boxplot theo nhóm nợ.
for patch, color in zip(box["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.92)
    patch.set_edgecolor("#4d4d4d")
# Duyệt từng đường trung vị để ghi giá trị median lên biểu đồ.
for i, median in enumerate(box["medians"]):
    x_coords = median.get_xdata()
    y_coords = median.get_ydata()
    x_pos = x_coords.mean()
    y_pos = y_coords[0] 
    plt.text(
        x_pos, 
        y_pos + 12,                  
        f"{int(y_pos)}",            
        ha="center", 
        va="bottom", 
        fontsize=10, 
        fontweight="bold", 
        color="#333333"             
    )
plt.title(" BIÊN ĐỘ CHI TIÊU THEO BIẾN ĐỊNH TÍNH ÁP LỰC VÀ NỢ", fontsize=16, fontweight="bold")
plt.xlabel("Mức độ Stress (Financial Stress Level)", fontsize=14)
plt.ylabel("Chi tiêu linh hoạt (Discretionary Spending)", fontsize=14)
plt.xticks([1, 2, 3], stress_order, fontsize=13)
plt.grid(axis="y", color="#cccccc", linewidth=1, alpha=0.9)
plt.legend(handles=[
    Patch(facecolor=red, edgecolor="#4d4d4d", label="Nguy hiểm (>35%)"),
    Patch(facecolor=blue, edgecolor="#4d4d4d", label="An toàn (<=35%)"),
], title="Cấu trúc nợ", loc="lower right", frameon=True, fontsize=12, title_fontsize=13)
save_chart("04_boxplot_no_stress.png")

# ============================================================
# 5. MA TRẬN TƯƠNG QUAN TUYẾN TÍNH ĐA BIẾN
# Mục tiêu: kiểm tra mối liên hệ tuyến tính giữa các biến tài chính.
# ============================================================
# 5. MA TRẬN TƯƠNG QUAN TUYẾN TÍNH ĐA BIẾN
# Danh sách các biến tài chính dùng để tính hệ số tương quan.
corr_columns = [
    "monthly_income", "monthly_expense_total", "savings_rate", "budget_goal", "credit_score",
    "debt_to_income_ratio", "loan_payment", "investment_amount", "emergency_fund",
    "transaction_count", "discretionary_spending", "essential_spending",
    "rent_or_mortgage", "actual_savings", "savings_goal_met",
]
# Tính ma trận tương quan Pearson giữa các biến số.
corr = tracker[corr_columns].corr(numeric_only=True)
corr.to_csv(out / "05_ma_tran_tuong_quan.csv", encoding="utf-8-sig")

plt.figure(figsize=(12, 10))
# Vẽ heatmap, màu đỏ thể hiện tương quan dương và màu xanh thể hiện tương quan âm.
image = plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(image, label="Hệ số tương quan")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.title(" MA TRẬN TƯƠNG QUAN TUYẾN TÍNH ĐA BIẾN", fontweight="bold")
# Duyệt từng ô trong ma trận để ghi hệ số tương quan lên heatmap.
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        value = corr.iloc[i, j]
        plt.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7,
                 color="white" if abs(value) > 0.55 else "black")
save_chart("05_heatmap_tuong_quan.png")

# ============================================================
# 6. MẬT ĐỘ PHÂN PHỐI ĐIỂM TÍN NHIỆM TRONG QUẦN THỂ
# Mục tiêu: xem phân phối điểm tín dụng và vùng điểm dưới ngưỡng chuẩn.
# ============================================================
# 6. MẬT ĐỘ PHÂN PHỐI ĐIỂM TÍN NHIỆM TRONG QUẦN THỂ
# Chuyển credit_score sang số và loại bỏ giá trị thiếu/vô cực.
credit = pd.to_numeric(tracker["credit_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
# Đặt ngưỡng tín dụng dưới chuẩn là 600 điểm.
subprime_threshold = 600
# Tính điểm tín dụng trung bình của toàn bộ hệ thống.
system_mean = credit.mean()
# Đếm số người có điểm tín dụng dưới ngưỡng 600.
below_count = int((credit < subprime_threshold).sum())
# Tính tỷ lệ phần trăm người dưới ngưỡng.
below_rate = below_count / len(credit) * 100

# Tạo bảng kết quả tóm tắt để lưu các chỉ số chính ra CSV.
pd.DataFrame({
    "metric": ["Điểm tín dụng trung bình hệ thống", "Ngưỡng tín dụng dưới chuẩn", "Số người dưới ngưỡng", "Tỷ lệ dưới ngưỡng (%)"],
    "value": [system_mean, subprime_threshold, below_count, below_rate]
}).to_csv(out / "06_phan_phoi_diem_tin_nhiem.csv", index=False, encoding="utf-8-sig")

# Tạo biểu đồ tạm để lấy dữ liệu tọa độ KDE từ pandas.
temporary_figure, temporary_axis = plt.subplots()
# Vẽ KDE tạm để pandas tính đường mật độ xác suất.
credit.plot(kind="kde", ax=temporary_axis)
line = temporary_axis.lines[0]
# Lấy tọa độ X của đường KDE.
x_data = line.get_xdata()
# Lấy tọa độ Y của đường KDE.
y_data = line.get_ydata()
plt.close(temporary_figure)

# Tạo khung biểu đồ lớn hơn để hiển thị 6 boxplot rõ ràng.
plt.figure(figsize=(12, 7))
plt.plot(x_data, y_data, color="#8e44ad", linewidth=3)
plt.fill_between(x_data, y_data, color="#8e44ad", alpha=0.22)
# Tạo mask để xác định phần đường KDE nằm dưới ngưỡng 600.
below_mask = x_data <= subprime_threshold
plt.fill_between(x_data[below_mask], y_data[below_mask], color="red", alpha=0.18, label="Vùng dưới ngưỡng (<600)")
plt.axvline(subprime_threshold, color="red", linestyle="--", linewidth=2.4, label=f"Ngưỡng tín dụng dưới chuẩn (<{subprime_threshold})")
plt.axvline(system_mean, color="blue", linestyle=":", linewidth=2.8, label=f"Trung bình hệ thống ({system_mean:.0f})")
plt.title(" MẬT ĐỘ PHÂN PHỐI ĐIỂM TÍN NHIỆM TRONG QUẦN THỂ", fontsize=16, fontweight="bold")
plt.xlabel("Điểm Tín Nhiệm (Credit Score)", fontsize=14)
plt.ylabel("Mật độ xác suất", fontsize=14)
plt.grid(color="#cccccc", linewidth=1, alpha=0.9)
plt.legend(loc="upper left", frameon=True, fontsize=12)
plt.text(0.72, 0.78, f"Số người dưới ngưỡng: {below_count}\nTỷ lệ: {below_rate:.2f}%",
         transform=plt.gca().transAxes, fontsize=11,
         bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray", alpha=0.85))
save_chart("06_kde_credit_score.png")

# ============================================================
# 7. TƯƠNG QUAN THU NHẬP VỚI MỨC ĐỘ VUNG TAY CHI TIÊU
# Mục tiêu: xem thu nhập tăng có đi kèm chi tiêu tăng hay không.
# ============================================================
# 7. TƯƠNG QUAN THU NHẬP VỚI MỨC ĐỘ VUNG TAY CHI TIÊU
# Tạo bảng thống kê theo nhóm đạt/không đạt mục tiêu tiết kiệm.
summary_7 = tracker.groupby("savings_goal_met", as_index=False).agg(
    count=("user_id", "count"),
    avg_income=("monthly_income", "mean"),
    avg_expense=("monthly_expense_total", "mean"),
    avg_savings_rate=("savings_rate", "mean"),
)
summary_7["status"] = summary_7["savings_goal_met"].map({0: "Không đạt mục tiêu tiết kiệm", 1: "Đạt mục tiêu tiết kiệm"})
summary_7.to_csv(out / "07_thu_nhap_va_bay_lam_phat_loi_song.csv", index=False, encoding="utf-8-sig")

# Lấy dữ liệu thu nhập, chi tiêu và trạng thái tiết kiệm để vẽ scatter plot.
scatter_df = tracker[["monthly_income", "monthly_expense_total", "savings_goal_met"]].replace([np.inf, -np.inf], np.nan).dropna()
# Gán thu nhập hàng tháng làm biến X.
x = scatter_df["monthly_income"].astype(float)
# Gán tổng chi tiêu hàng tháng làm biến Y.
y = scatter_df["monthly_expense_total"].astype(float)
# Tính hệ số tương quan giữa thu nhập và chi tiêu.
correlation = x.corr(y)
# Tính hệ số hồi quy tuyến tính bậc 1.
regression_coef = np.polyfit(x.to_numpy(), y.to_numpy(), 1)
line_x = np.linspace(x.min(), x.max(), 100)
line_y = regression_coef[0] * line_x + regression_coef[1]

plt.figure(figsize=(9, 6))
for value, color, label in [
    (0, "#e75c5c", "Không đạt mục tiêu tiết kiệm"),
    (1, "#3ca9e0", "Đạt mục tiêu tiết kiệm"),
]:
    subset = scatter_df[scatter_df["savings_goal_met"] == value]
    plt.scatter(subset["monthly_income"], subset["monthly_expense_total"], alpha=0.55, color=color, label=label)
plt.title(" TƯƠNG QUAN THU NHẬP VỚI MỨC ĐỘ VUNG TAY CHI TIÊU", fontweight="bold")
plt.xlabel("Thu nhập hàng tháng")
plt.ylabel("Tổng chi tiêu hàng tháng")
plt.legend(loc="upper left")
save_chart("07_scatter_income_vs_expense.png")

# ============================================================
# 8. TÓM TẮT KẾT QUẢ
# Mục tiêu: ghi các kết quả chính ra file TXT.
# ============================================================
# TÓM TẮT
# Tạo chuỗi văn bản tóm tắt các kết quả chính của 7 biểu đồ.
summary_text = f"""TÓM TẮT KẾT QUẢ

1. CÁN CÂN THANH KHOẢN TIỀN MẶT NGẮN HẠN
- Tổng thu nhập: {total_income:,.2f}
- Tổng chi tiêu: {total_expenses:,.2f}
- Dòng tiền ròng: {net_cash_flow:,.2f}
- Tỷ lệ tiêu dùng: {consumption_ratio:.2f}%

2. BIÊN ĐỘ LỆCH RÒNG GIỮA THỰC CHI VỚI HẠN MỨC NGÂN SÁCH
- Xanh lá: chi dưới ngân sách.
- Đỏ: vượt ngân sách.

3. DANH MỤC LŨY KẾ TIÊU TỐN TIỀN MẶT DÀI HẠN
- Danh mục chi lớn nhất: {long_term.iloc[0]["Category"]}
- Tổng chi: {long_term.iloc[0]["Amount"]:,.2f}
- Tỷ trọng: {long_term.iloc[0]["share_percent"]:.2f}%

4. BIÊN ĐỘ CHI TIÊU THEO BIẾN ĐỊNH TÍNH ÁP LỰC VÀ NỢ
- Đỏ: Nguy hiểm, DTI > 35%.
- Xanh dương: An toàn, DTI <= 35%.
- Trục X: Low - High - Medium theo mẫu.

5. MA TRẬN TƯƠNG QUAN TUYẾN TÍNH ĐA BIẾN
- Heatmap đã có số trong từng ô.

6. MẬT ĐỘ PHÂN PHỐI ĐIỂM TÍN NHIỆM TRONG QUẦN THỂ
- Ngưỡng tín dụng dưới chuẩn: < {subprime_threshold}.
- Trung bình hệ thống: {system_mean:.2f}.
- Số người dưới ngưỡng: {below_count}.
- Tỷ lệ dưới ngưỡng: {below_rate:.2f}%.

7. TƯƠNG QUAN THU NHẬP VỚI MỨC ĐỘ VUNG TAY CHI TIÊU
- Đã thêm đường hồi quy.
- Hệ số tương quan r = {correlation:.2f}.
"""
# Ghi nội dung tóm tắt ra file TXT.
(out / "tom_tat_ket_qua.txt").write_text(summary_text, encoding="utf-8")
# In thông báo hoàn thành khi chương trình chạy xong.
print("Hoàn thành tạo lại toàn bộ biểu đồ và bảng kết quả.")

XÂY DỰNG HỆ THỐNG HỌC MÁY LAI GHÉP ĐA TẦNG (HYBRID MACHINE LEARNING PIPELINE)

Trong bước này, chúng ta tiến hành nghiên cứu và triển khai một hệ thống học máy lai ghép hai tầng (Two-Tier Hybrid Machine Learning System) tích hợp kỹ thuật **Xếp chồng đặc trưng (Feature Stacking)** nâng cao:

1.  **Tầng 1 - Các chuyên gia phân loại (Classification Layer):** 
    *   Huấn luyện đồng thời 3 mô hình học máy: **Hồi quy Logistic (Logistic Regression)**, **Cây quyết định (Decision Tree)**, và **Rừng ngẫu nhiên cân bằng (Balanced Random Forest)**.
    *   Nhiệm vụ: Dự đoán khả năng một cá nhân có đạt mục tiêu tiết kiệm hay không (`savings_goal_met`).
    *   Đầu ra bổ trợ: Trích xuất ma trận xác suất phân phối xác định của từng lớp làm các biến phái sinh mới đại diện cho "độ tin cậy" của các chuyên gia phân loại.
2.  **Tầng trung gian - Xếp chồng đặc trưng (Feature Stacking):** 
    *   Nối (stack) các vector xác suất dự đoán sinh ra từ Tầng 1 trực tiếp vào ma trận đặc trưng gốc đã chuẩn hóa để tạo thành tập dữ liệu lai ghép (Hybrid Dataset).
3.  **Tầng 2 - Giám đốc hồi quy (Regression Meta-Model):** 
    *   Huấn luyện mô hình **Rừng ngẫu nhiên hồi quy (Random Forest Regressor)** quy mô lớn trên tập dữ liệu lai ghép.
    *   Nhiệm vụ: Đưa ra dự báo định lượng chính xác về số tiền tiết kiệm thực tế (`actual_savings`).

In [ ]:
import pandas as pd # Thư viện thao tác dữ liệu dạng bảng (Dataframe)
import numpy as np # Thư viện tính toán toán học và ma trận
import matplotlib.pyplot as plt # Thư viện vẽ đồ thị cơ bản
import seaborn as sns # Thư viện vẽ đồ thị thống kê đẹp mắt dựa trên matplotlib
import time # Thư viện đo thời gian thực thi
from pathlib import Path # Thư viện quản lý đường dẫn file/thư mục đa nền tảng

# Thư viện Scikit-Learn (Bộ công cụ Machine Learning)
from sklearn.model_selection import train_test_split # Hàm chia dữ liệu thành tập Train và Test
from sklearn.preprocessing import LabelEncoder, StandardScaler # Tiền xử lý: Mã hóa nhãn và Chuẩn hóa tỷ lệ dữ liệu
from sklearn.linear_model import LogisticRegression # Thuật toán Phân loại Hồi quy Logistic
from sklearn.tree import DecisionTreeClassifier # Thuật toán Phân loại Cây quyết định
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor # Thuật toán Rừng ngẫu nhiên
# Các hàm đo lường hiệu suất mô hình (Bổ sung classification_report và mean_squared_error)
from sklearn.metrics import classification_report, r2_score, mean_absolute_error, mean_squared_error

# =========================================================================
# 0. KHỞI ĐỘNG HỆ THỐNG VÀ CÀI ĐẶT CẤU HÌNH
# =========================================================================
print("[*] Đang khởi động hệ thống phân tích...")
global_start_time = time.time() # Bắt đầu đo tổng thời gian chạy script

# Thiết lập thẩm mỹ cho đồ thị
plt.rcParams['font.family'] = 'DejaVu Sans' # Thiết lập font chữ mặc định cho đồ thị
sns.set_theme(style="whitegrid") # Đặt nền đồ thị có lưới màu trắng để dễ nhìn
out_dir = Path("python_finance_outputs") # Khai báo thư mục lưu kết quả đầu ra
out_dir.mkdir(exist_ok=True) # Tạo thư mục nếu chưa có

# =========================================================================
# 1. TIỀN XỬ LÝ DỮ LIỆU (DATA PREPROCESSING)
# =========================================================================
try:
    df = pd.read_csv("Track_CLEANED.csv") # Đọc file dữ liệu csv
    print("[*] Đã nạp thành công dữ liệu gốc.")
except FileNotFoundError: # Xử lý ngoại lệ nếu không tìm thấy file để tránh chương trình bị crash
    print("Lỗi: Không tìm thấy file dữ liệu Track_CLEANED.csv!")
    exit() # Dừng chương trình nếu lỗi

# Lọc bỏ các dòng thiếu dữ liệu mục tiêu tiết kiệm
df = df.dropna(subset=['savings_goal_met', 'actual_savings'])

# 9 Biến đầu vào cơ sở (Base Features)
base_features = [
    "debt_to_income_ratio", "credit_score", "savings_rate", 
    "emergency_fund", "loan_payment", "monthly_income", 
    "monthly_expense_total", "discretionary_spending", "essential_spending"
]
# Lọc lại danh sách đặc trưng: Chỉ giữ lại những cột thực sự tồn tại trong Dataframe
base_features = [col for col in base_features if col in df.columns]

# Xử lý giá trị khuyết thiếu bằng Median
for col in base_features:
    if df[col].isnull().any():# Nếu cột có chứa giá trị NaN
        df[col] = df[col].fillna(df[col].median()) # Điền các ô trống bằng giá trị trung vị

X_base = df[base_features] # Ma trận chứa các biến đầu vào gốc

# Mục tiêu 1 cho Tầng Phân loại (0 = Không đạt, 1 = Đạt mục tiêu)
le = LabelEncoder() 
y_class = le.fit_transform(df['savings_goal_met']) 
target_names = le.classes_ # Lưu lại tên các nhãn gốc để dùng sau này

# Mục tiêu 2 cho Tầng Hồi quy (Dự đoán số tiền tiết kiệm thực tế)
y_reg = df['actual_savings']

# Chia tập Train/Test (Tỷ lệ 80/20, giữ phân tầng lớp)
X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split( 
    X_base, y_class, y_reg, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_class 
)

# Chuẩn hóa dữ liệu gốc (Z-score Scaling)
scaler = StandardScaler() 
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test) 

# =========================================================================
# 2. TẦNG 1: HUẤN LUYỆN VÀ ĐÁNH GIÁ CÁC "CHUYÊN GIA" PHÂN LOẠI
# =========================================================================
print("\n[VÒNG 1] ĐÁNH GIÁ CÁC MÔ HÌNH PHÂN LOẠI (CLASSIFICATION)")
print("-" * 70)

# Khai báo danh sách các mô hình học máy phân loại
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
    "Balanced Random Forest": RandomForestClassifier(n_estimators=200, class_weight='balanced', max_depth=10, random_state=42)
}

train_probs_list = [] 
test_probs_list = []
hybrid_feature_names = base_features.copy() # Khởi tạo danh sách tên biến lai ghép

# Duyệt qua từng mô hình để huấn luyện, đo thời gian và in báo cáo chuẩn
for name, model in models.items():
    model_start = time.time() # Bắt đầu bấm giờ cho riêng mô hình này
    
    # Huấn luyện mô hình phân loại dựa trên tập Train
    model.fit(X_train_scaled, y_class_train)
    y_pred = model.predict(X_test_scaled)
    
    model_end = time.time() # Dừng bấm giờ mô hình
    
    # Xuất báo cáo y hệt định dạng mong muốn
    print(f"▶ {name} Report:")
    print(classification_report(y_class_test, y_pred, digits=2))
    print(f"Thời gian thực thi: {model_end - model_start:.2f} giây")
    print("-" * 70)

    # Trích xuất xác suất dự đoán (%) để làm đầu vào cho Tầng 2
    train_probs_list.append(model.predict_proba(X_train_scaled))
    test_probs_list.append(model.predict_proba(X_test_scaled))

    # Đặt tên cột mới cho các thuộc tính xác suất vừa sinh ra
    for class_name in target_names:
        hybrid_feature_names.append(f"Prob_{name.replace(' ', '_')}_{class_name}")

# =========================================================================
# 3. GIAI ĐOẠN XẾP CHỒNG ĐẶC TRƯNG (FEATURE STACKING)
# =========================================================================
print("\n[*] Đang gộp dữ liệu: Bơm các cột xác suất vào dữ liệu gốc...")
X_train_hybrid = np.hstack([X_train_scaled] + train_probs_list)
X_test_hybrid = np.hstack([X_test_scaled] + test_probs_list)

# =========================================================================
# 4. TẦNG 2: HUẤN LUYỆN "GIÁM ĐỐC" HỒI QUY (META-MODEL)
# =========================================================================
print("\n[VÒNG 2] DỰ ĐOÁN SỐ TIỀN TIẾT KIỆM THỰC TẾ (REGRESSION)")
print("-" * 70)

meta_start = time.time() # Bấm giờ cho tầng hồi quy

# Khởi tạo mô hình Random Forest dạng Hồi quy gồm 300 cây
regressor = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42)
regressor.fit(X_train_hybrid, y_reg_train) # Huấn luyện dựa trên tập dữ liệu lai ghép (gốc + xác suất)
reg_preds = regressor.predict(X_test_hybrid) # Dự đoán trên tập kiểm tra

# Đánh giá Meta-Model bằng các chỉ số cốt lõi
r2 = r2_score(y_reg_test, reg_preds) 
mae = mean_absolute_error(y_reg_test, reg_preds) 
rmse = np.sqrt(mean_squared_error(y_reg_test, reg_preds)) # Sai số bình phương trung bình (RMSE) giúp bắt lỗi lệch lớn

meta_end = time.time()

print(f"▶ Meta-Model RandomForest Regressor:")
print(f" -> Hệ số xác định (R² Score)        : {r2*100:.2f}%")
print(f" -> Sai số tiền dự đoán trung bình (MAE) : ${mae:.2f}")
print(f" -> Sai số căn lề bình phương (RMSE)  : ${rmse:.2f}")
print(f"Thời gian thực thi: {meta_end - meta_start:.2f} giây")
print("-" * 70)

# =========================================================================
# 5. XUẤT ĐỒ THỊ BÁO CÁO KẾT QUẢ VÀ TỔNG KẾT
# =========================================================================
print("\n[*] Đang khởi tạo và xuất đồ thị phân tích...")

# --- Đồ thị 1: Biểu đồ phân tán (Thực tế vs AI dự đoán) ---
plt.figure(figsize=(7, 5))
plt.scatter(y_reg_test, reg_preds, alpha=0.6, color='#27ae60', edgecolor='w', s=60)
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], 'r--', lw=2.5)
plt.title('HÌNH 1: SỐ TIỀN TIẾT KIỆM THỰC TẾ vs AI DỰ ĐOÁN', fontweight='bold', pad=15)
plt.xlabel('Tiết kiệm thực tế (Actual Savings)', fontweight='bold')
plt.ylabel('Hệ thống AI dự đoán (Predicted Savings)', fontweight='bold')
plt.tight_layout()
plt.savefig(out_dir / 'savings_scatter_plot.png', dpi=300)
plt.close()

# --- Đồ thị 2: Đánh giá tầm quan trọng của các Biến ---
importances = regressor.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 8))
colors = []
for i in indices:
    name = hybrid_feature_names[i]
    if 'Prob_Logistic' in name: colors.append('#3498db') # Màu xanh biển cho Logistic
    elif 'Prob_Decision' in name: colors.append('#e67e22') # Màu cam cho Decision Tree
    elif 'Prob_Balanced' in name: colors.append('#e74c3c') # Màu đỏ cho Random Forest
    else: colors.append('#27ae60') # Màu xanh lá cho dữ liệu gốc

plt.barh(range(len(indices)), importances[indices], color=colors, edgecolor='black', height=0.6)
plt.yticks(range(len(indices)), [hybrid_feature_names[i] for i in indices], fontsize=9)
plt.title('HÌNH 2: TẦM QUAN TRỌNG CỦA CÁC YẾU TỐ ĐỐI VỚI TIẾT KIỆM', fontweight='bold', pad=15)
plt.xlabel('Tỷ lệ đóng góp vào quyết định cuối cùng (%)', fontweight='bold')

for index, value in enumerate(importances[indices]):
    plt.text(value + 0.001, index, f"{value*100:.1f}%", va='center', fontweight='bold', fontsize=8)
    
plt.tight_layout()
plt.savefig(out_dir / 'savings_feature_importance.png', dpi=300)
plt.close()

# CHỐT TỔNG THỜI GIAN CHẠY CHƯƠNG TRÌNH
global_end_time = time.time()
print(f"\n[OK] Toàn bộ quá trình xử lý hoàn tất thành công!")
print(f" -> Thư mục lưu đồ thị báo cáo: {out_dir}")
print(f" -> Tổng thời gian thực thi toàn luồng dữ liệu: {global_end_time - global_start_time:.2f} giây.")